# Unit 5 - Pyramids (ML-Agents PPO + Curiosity/RND)
Train a PPO agent with curiosity to solve the Pyramids environment using Unity ML-Agents.

**Run each cell in order.**

In [1]:
# Step 1: Clone ML-Agents repo
!git clone --depth 1 https://github.com/Unity-Technologies/ml-agents

Cloning into 'ml-agents'...
remote: Enumerating objects: 2434, done.
remote: Counting objects: 100% (2434/2434), done.
remote: Compressing objects: 100% (1940/1940), done.
remote: Total 2434 (delta 934), reused 1260 (delta 473), pack-reused 0 (from 0)
Receiving objects: 100% (2434/2434), 97.54 MiB | 26.54 MiB/s, done.
Resolving deltas: 100% (934/934), done.
Updating files: 100% (2352/2352), done.


In [8]:
# Step 2: Install Python 3.10.12 (required for ML-Agents compatibility)
!wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
!chmod +x Miniconda3-latest-Linux-x86_64.sh
!./Miniconda3-latest-Linux-x86_64.sh -b -f -p /usr/local

# Accept Anaconda Terms of Service to allow package installation
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

!conda install -q -y --prefix /usr/local python=3.10.12 ujson

import os
os.environ['PYTHONPATH'] = '/usr/local/lib/python3.10/site-packages/'
# Ensure /usr/local/bin is at the front of PATH so our new python/pip/mlagents are found
os.environ['PATH'] = '/usr/local/bin:' + os.environ['PATH']

--2026-03-17 13:34:27--  https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
Resolving repo.anaconda.com (repo.anaconda.com)... 104.16.191.158, 104.16.32.241, 2606:4700::6810:bf9e, ...
Connecting to repo.anaconda.com (repo.anaconda.com)|104.16.191.158|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 162067098 (155M) [application/octet-stream]
Saving to: ‘Miniconda3-latest-Linux-x86_64.sh’

Miniconda3-latest-L 100%[===================>] 154.56M   281MB/s    in 0.5s    

2026-03-17 13:34:28 (281 MB/s) - ‘Miniconda3-latest-Linux-x86_64.sh’ saved [162067098/162067098]

PREFIX=/usr/local
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please

In [9]:
# Step 3: Install ML-Agents
# We use /usr/local/bin/pip3 to ensure we are using the 3.10 environment
%cd /content/ml-agents
!/usr/local/bin/pip3 install -e ./ml-agents-envs
!/usr/local/bin/pip3 install -e ./ml-agents

/content/ml-agents
Obtaining file:///content/ml-agents/ml-agents-envs
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.7/721.7 kB 23.1 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.7/756.7 kB 32.0 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 131.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 140.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 68.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 122.4 MB/s 

In [4]:
# Step 4: Download Pyramids Linux executable
!mkdir -p ./training-envs-executables/linux
!wget "https://huggingface.co/spaces/unity/ML-Agents-Pyramids/resolve/main/Pyramids.zip" \
    -O ./training-envs-executables/linux/Pyramids.zip
!unzip -d ./training-envs-executables/linux/ ./training-envs-executables/linux/Pyramids.zip
!chmod -R 755 ./training-envs-executables/linux/Pyramids/Pyramids

--2026-03-17 13:34:04--  https://huggingface.co/spaces/unity/ML-Agents-Pyramids/resolve/main/Pyramids.zip
Resolving huggingface.co (huggingface.co)... 18.164.174.55, 18.164.174.118, 18.164.174.23, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.55|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/62b2c0284ea3eff279c966ab/2f754b6bf0b817493a77fafe582b7dde47fa25a476d3f8e68a24fdd4efcaa282?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27Pyramids.zip%3B+filename%3D%22Pyramids.zip%22%3B&response-content-type=application%2Fzip&Expires=1773758044&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiRXBvY2hUaW1lIjoxNzczNzU4MDQ0fX0sIlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjJiMmMwMjg0ZWEzZWZmMjc5Yzk2NmFiLzJmNzU0YjZiZjBiODE3NDkzYTc3ZmFmZTU4MmI3ZGRlNDdmYTI1YTQ3NmQzZjhlNjhhMjRmZGQ0ZWZjYWEyODJcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomcmVzcG9uc2UtY29udGVudC10eXBlPS

In [10]:
# Step 5: Train the agent (uses PyramidsRND config with curiosity)
!mlagents-learn ./config/ppo/PyramidsRND.yaml \
    --env=./training-envs-executables/linux/Pyramids/Pyramids \
    --run-id="Pyramids" \
    --no-graphics


            ┐  ╖
        ╓╖╬│╡  ││╬╖╖
    ╓╖╬│││││┘  ╬│││││╬╖
 ╖╬│││││╬╜        ╙╬│││││╖╖                               ╗╗╗
 ╬╬╬╬╖││╦╖        ╖╬││╗╣╣╣╬      ╟╣╣╬    ╟╣╣╣             ╜╜╜  ╟╣╣
 ╬╬╬╬╬╬╬╬╖│╬╖╖╓╬╪│╓╣╣╣╣╣╣╣╬      ╟╣╣╬    ╟╣╣╣ ╒╣╣╖╗╣╣╣╗   ╣╣╣ ╣╣╣╣╣╣ ╟╣╣╖   ╣╣╣
 ╬╬╬╬┐  ╙╬╬╬╬│╓╣╣╣╝╜  ╫╣╣╣╬      ╟╣╣╬    ╟╣╣╣ ╟╣╣╣╙ ╙╣╣╣  ╣╣╣ ╙╟╣╣╜╙  ╫╣╣  ╟╣╣
 ╬╬╬╬┐     ╙╬╬╣╣      ╫╣╣╣╬      ╟╣╣╬    ╟╣╣╣ ╟╣╣╬   ╣╣╣  ╣╣╣  ╟╣╣     ╣╣╣┌╣╣╜
 ╬╬╬╜       ╬╬╣╣      ╙╝╣╣╬      ╙╣╣╣╗╖╓╗╣╣╣╜ ╟╣╣╬   ╣╣╣  ╣╣╣  ╟╣╣╦╓    ╣╣╣╣╣
 ╙   ╓╦╖    ╬╬╣╣   ╓╗╗╖            ╙╝╣╣╣╣╝╜   ╘╝╝╜   ╝╝╝  ╝╝╝   ╙╣╣╣    ╟╣╣╣
   ╩╬╬╬╬╬╬╦╦╬╬╣╣╗╣╣╣╣╣╣╣╝                                             ╫╣╣╣╣
      ╙╬╬╬╬╬╬╬╣╣╣╣╣╣╝╜
          ╙╬╬╬╣╣╣╜
             ╙
        
 Version information:
  ml-agents: 1.2.0.dev0,
  ml-agents-envs: 1.2.0.dev0,
  Communicator API: 1.5.0,
  PyTorch: 2.8.0+cu128
[INFO] Connected to Unity environment with package version 2.2.1-exp.1 and communication version 1.5.0
[INFO] Connected new brain: Pyramids?team=0

In [12]:
# Step 6: Login to Hugging Face
from huggingface_hub import notebook_login
notebook_login()

In [13]:
# Step 7: Push model to Hugging Face Hub
!mlagents-push-to-hf \
    --run-id="Pyramids" \
    --local-dir="./results/Pyramids" \
    --repo-id="TheBestMoldyCheese/ppo-Pyramids" \
    --commit-message="Upload trained Pyramids PPO+RND agent - Unit 5 HF DRL Course"

[INFO] This function will create a model card and upload your Pyramids into HuggingFace Hub. This is a work in progress: If you encounter a bug, please send open an issue
[INFO] Pushing repo Pyramids to the Hugging Face Hub
Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  ...ids/Pyramids-1499965.onnx:   1% 13.0k/1.42M [00:00<?, ?B/s]


  ...amids/Pyramids-1499965.pt:   1% 79.4k/8.66M [00:00<?, ?B/s]



  ...ids/Pyramids-1999995.onnx:   1% 13.0k/1.42M [00:00<?, ?B/s]




  ...amids/Pyramids-1999995.pt:   1% 79.4k/8.66M [00:00<?, ?B/s]





  ...ids/Pyramids-2499969.onnx:   1% 13.0k/1.42M [00:00<?, ?B/s]






  ...amids/Pyramids-2499969.pt:   1% 79.4k/8.66M [00:00<?, ?B/s]







  ...ids/Pyramids-2999985.onnx:   1% 13.0k/1.42M [00:00<?, ?B/s]








  ...amids/Pyramids-2999985.pt:   1% 79.4k/8.66M [00:00<?, ?B/s]









  ...ids/Pyramids-3000113.onnx:   1% 13.0k/1.42M [00:00<?, ?B/s]
